# **NetMHCpan wrapper testing notebook**

In [1]:
import pandas as pd
import numpy as np
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))
from src.single_predictor import SinglePredictor
from src.pan_predictor import PanPredictor
from pathlib import Path
import subprocess

### **Testing single predictor class**

In [2]:
PATH_TO_NETMHCPAN = Path('/home/aalarkin/Work/netmhcpan_wrapper/netMHCpan-4.2bstatic.Linux/netMHCpan-4.2')
TMPDIR = Path('/home/aalarkin/Work/netmhcpan_wrapper/tmp')
PATH_TO_SUPERGROUPS = Path('/home/aalarkin/Work/netmhcpan_wrapper/datasets/mhc_supergroups.txt')
PATH_TO_MAPPING = Path('/home/aalarkin/Work/netmhcpan_wrapper/datasets/allele_nomenclature_mapping.txt')


In [3]:
single_predictor = SinglePredictor(
    path_to_netmhcpan=PATH_TO_NETMHCPAN,
    path_to_mapping=PATH_TO_MAPPING,
    tmpdir=TMPDIR,
    n_cores=8,
)

#testing on sample dataset
df = pd.read_csv('../datasets/chowell.txt', sep='\t')[:100]
df = single_predictor.match_hla(df, 'hla')
df = single_predictor.predict_affinity_dataframe(df, 'antigen.epitope', 'matched_hla')
display(df)


,antigen.epitope,hla,immunogenicity,matched_hla,%Rank Score_BA,%Rank_BA,Aff(nM)
0,ITDFNIDTY,HLA-A*01:01,Positive,HLA-A01:01,0.732399,0.008,18.09
1,ATDALMTGF,HLA-A*01:01,Positive,HLA-A01:01,0.427019,0.133,492.51
2,SSNIMSESY,HLA-A*01:01,Positive,HLA-A01:01,0.464926,0.095,326.81
3,VSEKYTDMY,HLA-A*01:01,Positive,HLA-A01:01,0.710627,0.009,22.90
4,FTDWANKQY,HLA-A*01:01,Positive,HLA-A01:01,0.715700,0.009,21.67
...,...,...,...,...,...,...,...
95,IADMGHLKY,HLA-A*01:01,Negative,HLA-A01:01,0.676161,0.011,33.24
96,DSDLRNDSY,HLA-A*01:01,Negative,HLA-A01:01,0.556244,0.040,121.67
97,ITEDKTELY,HLA-A*01:01,Negative,HLA-A01:01,0.718329,0.009,21.07
98,ALQEALTEHY,HLA-A*01:01,Negative,HLA-A01:01,0.259031,0.589,3032.42


In [4]:
#testing on data with inconsistent labels
inconsistent_data = {
    "IVDSLTEMY": "HLA-A*01:01",
    "LIDGIFLRY": "HLA-A*0101",
    "GLLPSLLLLL": "HLA-A0201",
    "EVISVMKRR": "A6801",
    "FQKVITEY": "B*15:01",
    "SEEDFIRSL": "B*4104",
    "VMADRTRHL": "E0103",
    "ANADLEVKI": "HLA-C*0602"
}
df = pd.DataFrame(list(inconsistent_data.items()), columns=['epitope', 'hla'])

df = single_predictor.match_hla(df, 'hla')
df = single_predictor.predict_affinity_dataframe(df, 'epitope', 'matched_hla')
display(df)



,epitope,hla,matched_hla,%Rank Score_BA,%Rank_BA,Aff(nM)
0,IVDSLTEMY,HLA-A*01:01,HLA-A01:01,0.658625,0.013,40.19
1,LIDGIFLRY,HLA-A*0101,HLA-A01:01,0.675019,0.011,33.66
2,GLLPSLLLLL,HLA-A0201,HLA-A02:01,0.752222,0.131,14.60
3,EVISVMKRR,A6801,HLA-A68:01,0.834083,0.041,6.02
4,FQKVITEY,B*15:01,HLA-B15:01,0.563743,0.624,112.19
5,SEEDFIRSL,B*4104,HLA-B41:04,0.578451,0.147,95.69
6,VMADRTRHL,E0103,HLA-E01:03,0.384711,0.032,778.44
7,ANADLEVKI,HLA-C*0602,HLA-C06:02,0.128117,5.632,12501.21


### **Testing pan predictor class**

In [5]:
predictor = PanPredictor(
    path_to_netmhcpan=PATH_TO_NETMHCPAN,
    path_to_supergroups=PATH_TO_SUPERGROUPS,
    tmpdir=TMPDIR,
    n_cores=16,
)

#no superfamily known
epitopes = ['LIDGIFLRY', 'VMADRTRHL', 'ANADLEVKI']
df = pd.DataFrame(epitopes, columns=['epitope'])
df = predictor.pan_hla_matching_prediction(df, 'epitope')
display(df)

#results are different from testing notebook as fixed logics bugs for supergroup matching


,epitope,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
0,LIDGIFLRY,SLA-1-YDL01,0.733159,0.009,17.94
1,VMADRTRHL,BoLA-6:01301,0.657554,0.227,40.66
2,ANADLEVKI,BoLA-3:01101,0.354021,0.418,1085.02


In [6]:
#superfamilies are known

data = {
    "LIDGIFLRY": "HLA-A01", #ref - hla a0101
    "FQKVITEY": "HLA-B15",    #ref - hla b1501
    "VMADRTRHL": "HLA-E01",      #ref - e0103
    "ANADLEVKI": "HLA-C06"       #ref - HLA-C*0602
}

df = pd.DataFrame(list(data.items()), columns=['epitope', 'family'])

df = predictor.pan_hla_matching_prediction(df, 'epitope', 'family')
display(df)


,epitope,family,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
0,LIDGIFLRY,HLA-A01,HLA-A01:191,0.715868,0.008,21.63
1,FQKVITEY,HLA-B15,HLA-B15:74,0.702537,0.433,24.99
2,VMADRTRHL,HLA-E01,HLA-E01:01,0.384711,0.032,778.44
3,ANADLEVKI,HLA-C06,HLA-C06:03,0.211832,4.290,5053.32


In [7]:
#some superfamilies are known

data = {
    "LIDGIFLRY": None,          #ref - hla a0101  
    "VMADRTRHL": "HLA-E01",      #ref - e0103
    "ANADLEVKI": "HLA-C06",       #ref - HLA-C*0602
                    
}

df = pd.DataFrame(list(data.items()), columns=['epitope', 'family'])

df = predictor.pan_hla_matching_prediction(df, 'epitope', 'family')
display(df)


,epitope,family,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
0,LIDGIFLRY,NaN,SLA-1-YDL01,0.733159,0.009,17.94
1,VMADRTRHL,HLA-E01,HLA-E01:01,0.384711,0.032,778.44
2,ANADLEVKI,HLA-C06,HLA-C06:03,0.211832,4.290,5053.32


###  **Testing final prediction function**

In [8]:
from src.run_netmhcpan import run_netmhcpan

In [9]:
df = pd.read_csv('../datasets/chowell.txt', sep='\t')[:100]
run_netmhcpan('single', PATH_TO_NETMHCPAN, df = df, epitope_colname='antigen.epitope', hla_column= 'hla')

,antigen.epitope,hla,immunogenicity,matched_hla,%Rank Score_BA,%Rank_BA,Aff(nM)
0,ITDFNIDTY,HLA-A*01:01,Positive,HLA-A01:01,0.732399,0.008,18.09
1,ATDALMTGF,HLA-A*01:01,Positive,HLA-A01:01,0.427019,0.133,492.51
2,SSNIMSESY,HLA-A*01:01,Positive,HLA-A01:01,0.464926,0.095,326.81
3,VSEKYTDMY,HLA-A*01:01,Positive,HLA-A01:01,0.710627,0.009,22.90
4,FTDWANKQY,HLA-A*01:01,Positive,HLA-A01:01,0.715700,0.009,21.67
...,...,...,...,...,...,...,...
95,IADMGHLKY,HLA-A*01:01,Negative,HLA-A01:01,0.676161,0.011,33.24
96,DSDLRNDSY,HLA-A*01:01,Negative,HLA-A01:01,0.556244,0.040,121.67
97,ITEDKTELY,HLA-A*01:01,Negative,HLA-A01:01,0.718329,0.009,21.07
98,ALQEALTEHY,HLA-A*01:01,Negative,HLA-A01:01,0.259031,0.589,3032.42


In [10]:
data = {
    "LIDGIFLRY": None,          #ref - hla a0101  
    "VMADRTRHL": "HLA-E01",      #ref - e0103
    "ANADLEVKI": "HLA-C06",       #ref - HLA-C*0602
                    
}

df = pd.DataFrame(list(data.items()), columns=['epitope', 'family'])

run_netmhcpan('pan', PATH_TO_NETMHCPAN, df, epitope_colname='epitope', supergroup_column='family')

,epitope,family,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
0,LIDGIFLRY,NaN,SLA-1-YDL01,0.733159,0.009,17.94
1,VMADRTRHL,HLA-E01,HLA-E01:01,0.384711,0.032,778.44
2,ANADLEVKI,HLA-C06,HLA-C06:03,0.211832,4.290,5053.32
